# EfficientNet-B0 FER2013 Experiments

Focused transfer-learning notebook for EfficientNet-B0. This keeps EfficientNet experiments separate from ResNet18 so results and configs are easier to report.

In [ ]:
from pathlib import Path
import json
import random
import sys

import importlib
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'fer2013_images'
# Kaggle option:
# DATA_DIR = Path('/kaggle/input/datasets/msambare/fer2013')

RESULTS_DIR = PROJECT_ROOT / 'results'
(RESULTS_DIR / 'checkpoints').mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'metrics').mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'figures').mkdir(parents=True, exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE, DATA_DIR

## Kaggle Path Setup Option

Use this block only when running the notebook in Kaggle. It clones or updates the repository, then points `DATA_DIR` to the Kaggle FER2013 dataset folder.

In [ ]:
# Kaggle option. Uncomment this block only when running in Kaggle.

# %cd /kaggle/working
# from pathlib import Path
# import sys

# REPO_URL = 'https://github.com/zsykk/DL-final-project.git'
# REPO_DIR = 'DL-final-project'

# if Path(REPO_DIR).exists():
#     %cd /kaggle/working/DL-final-project
#     !git pull
# else:
#     !git clone {REPO_URL}
#     %cd /kaggle/working/DL-final-project

# PROJECT_ROOT = Path('/kaggle/working/DL-final-project')
# sys.path.insert(0, str(PROJECT_ROOT / 'src'))
# DATA_DIR = Path('/kaggle/input/datasets/msambare/fer2013')
# RESULTS_DIR = PROJECT_ROOT / 'results'
# (RESULTS_DIR / 'checkpoints').mkdir(parents=True, exist_ok=True)
# (RESULTS_DIR / 'metrics').mkdir(parents=True, exist_ok=True)
# (RESULTS_DIR / 'figures').mkdir(parents=True, exist_ok=True)

# DATA_DIR, (DATA_DIR / 'train').exists(), (DATA_DIR / 'test').exists(), PROJECT_ROOT

In [ ]:
from fer_project.data import TransformConfig, build_imagefolder_dataloaders, class_weights, dataset_labels
from fer_project.models import build_model

import fer_project.training as training
training = importlib.reload(training)
fit = training.fit

import fer_project.metrics as metrics
metrics = importlib.reload(metrics)
collect_predictions = metrics.collect_predictions
plot_training_and_confusion = metrics.plot_training_and_confusion
save_classification_report = metrics.save_classification_report
top_confusions = metrics.top_confusions

In [ ]:
TRANSFER_TRAIN_CONFIG = TransformConfig(image_size=224, channels=3, augment=True, imagenet_norm=True)
TRANSFER_EVAL_CONFIG = TransformConfig(image_size=224, channels=3, augment=False, imagenet_norm=True)

def save_experiment_config(name, config):
    path = RESULTS_DIR / 'metrics' / f'{name}_config.json'
    with open(path, 'w') as f:
        json.dump(config, f, indent=2, default=str)
    return path

def build_transfer_loaders(batch_size=64, num_workers=0, subset_fraction=1.0, weighted_sampler=False):
    return build_imagefolder_dataloaders(
        DATA_DIR,
        train_config=TRANSFER_TRAIN_CONFIG,
        eval_config=TRANSFER_EVAL_CONFIG,
        batch_size=batch_size,
        num_workers=num_workers,
        weighted_sampler=weighted_sampler,
        val_fraction=0.1,
        subset_fraction=subset_fraction,
        seed=SEED,
    )

def set_efficientnet_b0_trainable_layers(model, unfreeze_from='classifier'):
    for parameter in model.parameters():
        parameter.requires_grad = False
    if unfreeze_from == 'classifier':
        modules = [model.classifier]
    elif unfreeze_from == 'last_block':
        modules = [model.features[-1], model.classifier]
    elif unfreeze_from == 'last_blocks':
        modules = [model.features[-2], model.features[-1], model.classifier]
    elif unfreeze_from == 'last_three_blocks':
        modules = [model.features[-3], model.features[-2], model.features[-1], model.classifier]
    elif unfreeze_from == 'all':
        modules = [model]
    else:
        raise ValueError("unfreeze_from must be 'classifier', 'last_block', 'last_blocks', 'last_three_blocks', or 'all'")
    for module in modules:
        for parameter in module.parameters():
            parameter.requires_grad = True
    return model

def finalize_model_run(model, loaders, history, name, checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    y_true, y_pred = collect_predictions(model, loaders['test'], DEVICE)
    report = save_classification_report(y_true, y_pred, RESULTS_DIR / 'metrics' / f'{name}_classification_report.csv')
    confusions = top_confusions(y_true, y_pred, top_n=10)
    confusions.to_csv(RESULTS_DIR / 'metrics' / f'{name}_top_confusions.csv', index=False)
    history_frame = pd.DataFrame(history)
    history_frame.to_csv(RESULTS_DIR / 'metrics' / f'{name}_history.csv', index=False)
    plot_training_and_confusion(
        history_frame,
        y_true,
        y_pred,
        f'{name}: loss and confusion matrix',
        RESULTS_DIR / 'figures' / f'{name}_confusion_matrix.png',
    )
    print(f'Best checkpoint saved to: {checkpoint_path}')
    display(report.loc[['macro avg', 'weighted avg']])
    display(report.loc[['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral'], ['precision', 'recall', 'f1-score', 'support']])
    display(confusions)
    return history_frame, report, confusions

## Gradual Unfreeze

Trains EfficientNet-B0 in stages: classifier first, then progressively later feature blocks. Total epochs = `len(unfreeze_stages) * epochs_per_stage`.

In [ ]:
def fit_efficientnet_b0_gradual_unfreeze(
    model,
    loaders,
    device,
    unfreeze_stages=('classifier', 'last_block', 'last_blocks', 'last_three_blocks', 'all'),
    epochs_per_stage=5,
    lr_by_stage=None,
    weight_decay=1e-4,
    checkpoint_path=None,
):
    lr_by_stage = lr_by_stage or {
        'classifier': 1e-3,
        'last_block': 5e-4,
        'last_blocks': 1e-4,
        'last_three_blocks': 5e-5,
        'all': 1e-5,
    }
    model = model.to(device)
    criterion = training.build_criterion(loss_name='cross_entropy')
    history = []
    best_val_macro_f1 = float('-inf')
    best_val_loss = float('inf')
    global_epoch = 0
    for stage_name in unfreeze_stages:
        if checkpoint_path is not None and Path(checkpoint_path).exists():
            model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        set_efficientnet_b0_trainable_layers(model, stage_name)
        trainable_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable_params, lr=lr_by_stage[stage_name], weight_decay=weight_decay)
        for stage_epoch in range(1, epochs_per_stage + 1):
            global_epoch += 1
            train_metrics = training.train_one_epoch(model, loaders['train'], criterion, optimizer, device)
            val_metrics = training.evaluate_loss_f1(model, loaders['val'], criterion, device)
            row = {
                'stage': f'unfreeze_{stage_name}',
                'epoch': global_epoch,
                'stage_epoch': stage_epoch,
                'lr': optimizer.param_groups[0]['lr'],
                'train_loss': train_metrics['loss'],
                'train_macro_f1': train_metrics['macro_f1'],
                'train_weighted_f1': train_metrics['weighted_f1'],
                'val_loss': val_metrics['loss'],
                'val_macro_f1': val_metrics['macro_f1'],
                'val_weighted_f1': val_metrics['weighted_f1'],
            }
            history.append(row)
            if training.should_log_epoch(stage_epoch, epochs_per_stage):
                print(training.format_epoch_metrics(row))
            if checkpoint_path is not None and training.is_better_checkpoint(val_metrics, best_val_macro_f1, best_val_loss):
                best_val_macro_f1 = val_metrics['macro_f1']
                best_val_loss = val_metrics['loss']
                Path(checkpoint_path).parent.mkdir(parents=True, exist_ok=True)
                torch.save(model.state_dict(), checkpoint_path)
    if checkpoint_path is not None and Path(checkpoint_path).exists():
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    return history

def run_efficientnet_b0_gradual_unfreeze_experiment(
    name='efficientnet_b0_gradual_unfreeze',
    batch_size=64,
    num_workers=0,
    subset_fraction=1.0,
    unfreeze_stages=('classifier', 'last_block', 'last_blocks', 'last_three_blocks', 'all'),
    epochs_per_stage=5,
):
    loaders, datasets = build_transfer_loaders(
        batch_size=batch_size,
        num_workers=num_workers,
        subset_fraction=subset_fraction,
        weighted_sampler=False,
    )
    model = build_model('transfer', transfer_model='efficientnet_b0', freeze_backbone=True)
    checkpoint_path = RESULTS_DIR / 'checkpoints' / f'{name}.pt'
    history = fit_efficientnet_b0_gradual_unfreeze(
        model,
        loaders,
        device=DEVICE,
        unfreeze_stages=unfreeze_stages,
        epochs_per_stage=epochs_per_stage,
        checkpoint_path=checkpoint_path,
    )
    save_experiment_config(name, {
        'name': name,
        'model': 'efficientnet_b0',
        'selection_rule': 'highest validation macro F1; lower validation loss tie-breaker',
        'batch_size': batch_size,
        'subset_fraction': subset_fraction,
        'unfreeze_stages': list(unfreeze_stages),
        'epochs_per_stage': epochs_per_stage,
        'total_epochs': len(unfreeze_stages) * epochs_per_stage,
        'train_transform': '224x224 RGB, augment=True, ImageNet normalization',
        'eval_transform': '224x224 RGB, augment=False, ImageNet normalization',
        'checkpoint': checkpoint_path,
    })
    return finalize_model_run(model, loaders, history, name, checkpoint_path)

history_effnet_unfreeze, report_effnet_unfreeze, confusions_effnet_unfreeze = run_efficientnet_b0_gradual_unfreeze_experiment()

## One-Layer Classifier Head

Loads the gradual-unfreeze checkpoint, replaces the head with `1280 -> 256 -> 7`, and fine-tunes the last EfficientNet feature blocks plus the classifier.

In [ ]:
def run_efficientnet_b0_small_head(
    name='efficientnet_b0_small_classifier',
    source_checkpoint=RESULTS_DIR / 'checkpoints' / 'efficientnet_b0_gradual_unfreeze.pt',
    batch_size=64,
    num_workers=0,
    subset_fraction=1.0,
    epochs=10,
    lr=5e-5,
    unfreeze_from='last_blocks',
    classifier_hidden_layers=[256],
):
    loaders, datasets = build_transfer_loaders(batch_size=batch_size, num_workers=num_workers, subset_fraction=subset_fraction)
    model = build_model('transfer', transfer_model='efficientnet_b0', freeze_backbone=False)
    model.load_state_dict(torch.load(source_checkpoint, map_location=DEVICE))
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = torch.nn.Sequential(
        torch.nn.Linear(in_features, classifier_hidden_layers[0]),
        torch.nn.BatchNorm1d(classifier_hidden_layers[0]),
        torch.nn.ReLU(inplace=True),
        torch.nn.Dropout(0.4),
        torch.nn.Linear(classifier_hidden_layers[0], 7),
    )
    set_efficientnet_b0_trainable_layers(model, unfreeze_from)
    checkpoint_path = RESULTS_DIR / 'checkpoints' / f'{name}.pt'
    history = fit(model, loaders, device=DEVICE, epochs=epochs, lr=lr, checkpoint_path=checkpoint_path)
    save_experiment_config(name, {
        'name': name,
        'model': 'efficientnet_b0',
        'source_checkpoint': source_checkpoint,
        'classifier_hidden_layers': classifier_hidden_layers,
        'classifier_shape': '1280 -> 256 -> 7',
        'unfreeze_from': unfreeze_from,
        'epochs': epochs,
        'lr': lr,
        'selection_rule': 'highest validation macro F1; lower validation loss tie-breaker',
        'checkpoint': checkpoint_path,
    })
    return finalize_model_run(model, loaders, history, name, checkpoint_path)

history_effnet_small_head, report_effnet_small_head, confusions_effnet_small_head = run_efficientnet_b0_small_head()

## Controlled Imbalance and Loss Tuning

These experiments reload the same EfficientNet-B0 one-layer classifier checkpoint and fine-tune the last blocks. Only the imbalance/loss strategy changes.

### Class Weights

In [ ]:
def run_efficientnet_b0_controlled_tune(
    name='efficientnet_b0_small_classifier_weighted_sampler_focal',
    source_checkpoint=RESULTS_DIR / 'checkpoints' / 'efficientnet_b0_small_classifier.pt',
    batch_size=64,
    num_workers=0,
    subset_fraction=1.0,
    epochs=15,
    lr=5e-5,
    unfreeze_from='last_blocks',
    classifier_hidden_layers=[256],
    use_class_weights=False,
    weighted_sampler=True,
    loss_name='focal',
    focal_gamma=1.0,
):
    loaders, datasets = build_transfer_loaders(
        batch_size=batch_size,
        num_workers=num_workers,
        subset_fraction=subset_fraction,
        weighted_sampler=weighted_sampler,
    )
    model = build_model(
        'transfer',
        transfer_model='efficientnet_b0',
        freeze_backbone=False,
        classifier_hidden_layers=classifier_hidden_layers,
        classifier_dropout=0.4,
    )
    model.load_state_dict(torch.load(source_checkpoint, map_location=DEVICE))
    set_efficientnet_b0_trainable_layers(model, unfreeze_from)
    weights = class_weights(dataset_labels(datasets['train'])) if use_class_weights else None
    checkpoint_path = RESULTS_DIR / 'checkpoints' / f'{name}.pt'
    history = fit(
        model,
        loaders,
        device=DEVICE,
        epochs=epochs,
        lr=lr,
        class_weight=weights,
        loss_name=loss_name,
        focal_gamma=focal_gamma,
        checkpoint_path=checkpoint_path,
    )
    save_experiment_config(name, {
        'name': name,
        'model': 'efficientnet_b0',
        'source_checkpoint': source_checkpoint,
        'classifier_hidden_layers': classifier_hidden_layers,
        'classifier_shape': '1280 -> 256 -> 7',
        'unfreeze_from': unfreeze_from,
        'epochs': epochs,
        'lr': lr,
        'use_class_weights': use_class_weights,
        'weighted_sampler': weighted_sampler,
        'loss_name': loss_name,
        'focal_gamma': focal_gamma,
        'selection_rule': 'highest validation macro F1; lower validation loss tie-breaker',
        'checkpoint': checkpoint_path,
    })
    return finalize_model_run(model, loaders, history, name, checkpoint_path)

history_effnet_class_weights, report_effnet_class_weights, confusions_effnet_class_weights = run_efficientnet_b0_controlled_tune(
    name='efficientnet_b0_small_classifier_class_weights',
    use_class_weights=True,
    weighted_sampler=False,
    loss_name='cross_entropy',
    focal_gamma=1.0,
)

### WeightedRandomSampler

In [ ]:
history_effnet_weighted_sampler, report_effnet_weighted_sampler, confusions_effnet_weighted_sampler = run_efficientnet_b0_controlled_tune(
    name='efficientnet_b0_small_classifier_weighted_sampler',
    use_class_weights=False,
    weighted_sampler=True,
    loss_name='cross_entropy',
    focal_gamma=1.0,
)

### Focal Loss

In [ ]:
history_effnet_focal, report_effnet_focal, confusions_effnet_focal = run_efficientnet_b0_controlled_tune(
    name='efficientnet_b0_small_classifier_focal',
    use_class_weights=False,
    weighted_sampler=False,
    loss_name='focal',
    focal_gamma=1.0,
)

### Class Weights + Focal Loss

In [ ]:
history_effnet_class_weights_focal, report_effnet_class_weights_focal, confusions_effnet_class_weights_focal = run_efficientnet_b0_controlled_tune(
    name='efficientnet_b0_small_classifier_class_weights_focal',
    use_class_weights=True,
    weighted_sampler=False,
    loss_name='focal',
    focal_gamma=1.0,
)

### WeightedRandomSampler + Focal Loss

In [ ]:
history_effnet_weighted_sampler_focal, report_effnet_weighted_sampler_focal, confusions_effnet_weighted_sampler_focal = run_efficientnet_b0_controlled_tune(
    name='efficientnet_b0_small_classifier_weighted_sampler_focal',
    use_class_weights=False,
    weighted_sampler=True,
    loss_name='focal',
    focal_gamma=1.0,
)

## Merge EfficientNet-B0 Results

Run this after finishing several EfficientNet-B0 cells. It reads saved metric files and creates one comparison table without retraining.

In [ ]:
summary_names = [
    'efficientnet_b0_gradual_unfreeze',
    'efficientnet_b0_small_classifier',
    'efficientnet_b0_small_classifier_class_weights',
    'efficientnet_b0_small_classifier_weighted_sampler',
    'efficientnet_b0_small_classifier_focal',
    'efficientnet_b0_small_classifier_class_weights_focal',
    'efficientnet_b0_small_classifier_weighted_sampler_focal',
]

summary_rows = {}
metrics_dir = RESULTS_DIR / 'metrics'
for name in summary_names:
    report_path = metrics_dir / f'{name}_classification_report.csv'
    history_path = metrics_dir / f'{name}_history.csv'
    confusions_path = metrics_dir / f'{name}_top_confusions.csv'
    if not report_path.exists() or not history_path.exists():
        continue
    report = pd.read_csv(report_path, index_col=0)
    history = pd.read_csv(history_path)
    confusions = pd.read_csv(confusions_path) if confusions_path.exists() else pd.DataFrame()
    summary_rows[name] = {
        'final_val_loss': float(history['val_loss'].iloc[-1]),
        'best_val_loss': float(history['val_loss'].min()),
        'final_val_macro_f1': float(history['val_macro_f1'].iloc[-1]),
        'best_val_macro_f1': float(history['val_macro_f1'].max()),
        'test_macro_f1': float(report.loc['macro avg', 'f1-score']),
        'test_weighted_f1': float(report.loc['weighted avg', 'f1-score']),
        'top_confusion': 'none' if confusions.empty else f"{confusions.iloc[0]['true_emotion']} -> {confusions.iloc[0]['predicted_emotion']}",
    }

efficientnet_b0_summary = pd.DataFrame(summary_rows).T.sort_values('test_macro_f1', ascending=False)
efficientnet_b0_summary.to_csv(metrics_dir / 'efficientnet_b0_experiment_summary.csv')
efficientnet_b0_summary